In [ ]:
from langchain.chat_models import ChatOpenAI
from typing import List, TypedDict, Type
from langchain.tools import BaseTool
from pydantic import BaseModel, Field
from langchain.agents import initialize_agent, AgentType
from langchain.agents.agent_toolkits import SQLDatabaseToolkit
from langchain.sql_database import SQLDatabase

import matplotlib.pyplot as plt
from dotenv import load_dotenv
load_dotenv()

class PlotDataArgsSchema(BaseModel):
    data: List[int] = Field(
        description="그래프에 들어갈 데이터 입니다.",
    )

class PlotGraphTool(BaseTool):
    name = "PlotGraphTool"
    description = """
    이 툴은 주어진 데이터를 이용하여 line plot을 그려줍니다.
    """
    args_schema: Type[PlotDataArgsSchema] = PlotDataArgsSchema

    def _run(self, data):
        plt.figure(figsize=(10, 6))
        plt.plot(data, label='Sample Data')
        plt.title('Sample Graph')
        plt.xlabel('X-axis')
        plt.ylabel('Y-axis')
        plt.legend()
        plt.grid(True)
        plt.show()

llm = ChatOpenAI(
    temperature=0.1, 
    model_name='gpt-4o-mini', 
)

agent = initialize_agent(
    llm=llm,
    verbose=True,
    agent=AgentType.OPENAI_FUNCTIONS,
    handle_parsing_errors=True,
    tools=[
        PlotGraphTool(),
    ],
)

agent.run("너는 이 데이터로 그래프를 그려줘야 합니다. '1, 2, 3, 4, 5, 6, 7, 8, 9, 10'")

## 웹검색, DB, PDF 등등 같이 사용하기

In [18]:
from langchain.agents import create_sql_agent, create_react_agent, Tool, AgentExecutor, AgentType, create_openai_tools_agent
from langchain.chat_models import ChatOpenAI
from langchain.sql_database import SQLDatabase
from langchain.agents.agent_toolkits import SQLDatabaseToolkit
from langchain.utilities import GoogleSerperAPIWrapper
from langchain.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
from langchain.document_loaders import PyPDFLoader
from langchain.tools.retriever import create_retriever_tool
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

from langchain_core.tools import tool

from typing import Annotated, List, Type
from langchain.tools import BaseTool
from pydantic import BaseModel, Field
import matplotlib.pyplot as plt
from dotenv import load_dotenv
# from langchain.chat_models import ChatOllama


load_dotenv()


# SQL Agent 생성
llm = ChatOpenAI(
    temperature=0.1, 
    model_name='gpt-4o-mini', 
)


# llm = ChatOllama(
#     model="llama3.1:latest",
#     temperature=0.1, 
# )

# 데이터베이스 설정
db = SQLDatabase.from_uri("sqlite:///movies.sqlite")
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

google_search = GoogleSerperAPIWrapper()

template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question. and TRANSLATE to korean

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(template)


prompt2=ChatPromptTemplate.from_messages(
    [
        ("system", "너는 테스트 챗봇이고 한국어로 답해줘.."),
        MessagesPlaceholder("chat_history", optional=True),
        ("human", "{input}"),
        MessagesPlaceholder("agent_scratchpad"),
    ]
)


# SQL Agent 생성 
sql_agent = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    agent_type=AgentType.OPENAI_FUNCTIONS,
    verbose=True,
)

### 1-2. PDF 문서 검색 도구 (Retriever) ###
# PDF 파일 로드. 파일의 경로 입력
# loader = PyPDFLoader("SPRi AI Brief_8월호_산업동향.pdf")
loader = PyPDFLoader("SPRI_AI_Brief_2023년12월호_F.pdf")

# 텍스트 분할기를 사용하여 문서를 분할합니다.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

# 문서를 로드하고 분할합니다.
split_docs = loader.load_and_split(text_splitter)

# VectorStore를 생성합니다.
vector = FAISS.from_documents(split_docs, OpenAIEmbeddings())

# Retriever를 생성합니다.
retriever = vector.as_retriever()

# langchain 패키지의 tools 모듈에서 retriever 도구를 생성
retriever_tool = create_retriever_tool(
    retriever,
    name="pdf search",
    # 도구에 대한 설명을 자세히 기입해야 합니다!!!
    description="'AI 산업동향' 과 관련된 질문은 이 도구를 사용해야 합니다!",
)

class PlotDataArgsSchema(BaseModel):
    data: List[int] = Field(
        description="정수형 list 데이터로 구성됩니다.",
    )

class PlotGraphTool(BaseTool):
    name = "PlotGraphTool"
    description = """
    이 툴은 주어진 데이터를 이용하여 line plot을 그려줍니다.
    """
    args_schema: Type[BaseModel] = PlotDataArgsSchema

    def _run(self, data):
        plt.figure(figsize=(10, 6))
        plt.plot(data, label='Sample Data')
        plt.title('Sample Graph')
        plt.xlabel('X-axis')
        plt.ylabel('Y-axis')
        plt.legend()
        plt.grid(True)
        plt.show()

@tool
def PlotGraphToolTool(data: Annotated[str, "데이터"]):
    """그래프를 그려야할 경우 이 도구를 사용하면 됩니다.
    문자열로 들어올 경우 수치형으로 변환해주세요. 
    
    Args:
        data (List[int]): 1,2,3,4,5,6,7
    """
    
    plt.figure(figsize=(10, 6))
    plt.plot(data, label='Sample Data')
    plt.title('Sample Graph')
    plt.xlabel('X-axis')
    plt.ylabel('Y-axis')
    plt.legend()
    plt.grid(True)
    plt.show()

react_agent_tools = [
    Tool(
        name="Intermediate_Answer",
        func=google_search.run,
        description="웹 검색이 필요할 때 사용합니다.",
        verbose=True
    ),
    Tool(
        name="Database_Search_for_movie",
        func=sql_agent.run,
        description="영화 관련 내용을 찾을 때 사용합니다.",
        verbose=True
    ),
    Tool(
        name="AI_Industry_Trends_PDF_Search",
        func=retriever_tool.run,
        description="'AI 산업 동향' 과 관련된 질문은 이 도구를 사용해야 합니다!",
        verbose=True
    ),
    PlotGraphToolTool
    # Tool(
    #     name="Generate_Graph",
    #     func=PlotGraphTool(),
    #     description="그래프를 그려야할 경우 이 도구를 사용하면 됩니다.",
    #     verbose=True
    # ),
]

# React Agent 생성
react_agent = create_openai_tools_agent(
    llm, 
    react_agent_tools, 
    prompt2, 
)

agent_executor = AgentExecutor(
    agent=react_agent,
    tools=react_agent_tools,
    handle_parsing_errors=True,
    verbose=True,
    return_intermediate_steps=True,
)

########## 6. 채팅 기록을 수행하는 메모리를 추가합니다. ##########

# 채팅 메시지 기록을 관리하는 객체를 생성합니다.
message_history = ChatMessageHistory()

# 채팅 메시지 기록이 추가된 에이전트를 생성합니다.
agent_with_chat_history = RunnableWithMessageHistory(
    agent_executor,
    # 대부분의 실제 시나리오에서 세션 ID가 필요하기 때문에 이것이 필요합니다
    # 여기서는 간단한 메모리 내 ChatMessageHistory를 사용하기 때문에 실제로 사용되지 않습니다
    lambda session_id: message_history,
    # 프롬프트의 질문이 입력되는 key: "input"
    input_messages_key="input",
    # 프롬프트의 메시지가 입력되는 key: "chat_history"
    history_messages_key="chat_history",
)

########## 7. 질의-응답 테스트를 수행합니다. ##########

# 질의에 대한 답변을 출력합니다.
response = agent_with_chat_history.invoke(
    {
        "input": "다음 데이터로 그래프를 그려줘 1,2,3,4,5,6,7"
    },
    # 세션 ID를 설정합니다.
    # 여기서는 간단한 메모리 내 ChatMessageHistory를 사용하기 때문에 실제로 사용되지 않습니다
    config={"configurable": {"session_id": "MyTestSessionID"}},
)
print(f"답변: {response['output']}")


        # "input": "YouTube 2024년부터 AI 생성콘텐츠 표시 의무화에 대한 내용을 PDF 문서에서 알려줘" 
        # "input": "Tell me the actors who starred in the movie with the highest burget"



> Entering new AgentExecutor chain...


TypeError: additional_kwargs["function"] already exists in this message, but with a different type.